# 5 — Sample-level response representations

## Questions

1. What does conventional cell-state composition show?
2. What do conventional CD4/CD8 functional gene programs show?
3. Do pooled scGPT embeddings add a complementary response pattern?

All representations are frozen before response is revealed. Response is shown
by color; the two samples from P1076 use triangles while all others use circles.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures/demo"
TABLES = RESULTS / "tables"
EMBEDDINGS = RESULTS / "embeddings"
for directory in (FIGURES, TABLES, EMBEDDINGS):
    directory.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA / "processed/GSE205335_phase1_raw_counts.h5ad"
RANDOM_STATE = 0

from itertools import combinations
from sklearn.metrics import pairwise_distances, silhouette_score

SCGPT_PATH = EMBEDDINGS / "GSE205335_scgpt_frozen_v1.h5ad"
assert SCGPT_PATH.exists(), "Run Notebook 3 first"
embedding = sc.read_h5ad(SCGPT_PATH)
source = sc.read_h5ad(RAW_PATH, backed="r")
meta = source.obs.loc[embedding.obs_names, ["Sample", "Patient", "Tissue origin", "Platform", "lineage.sub", "celltype"]].copy()
response_locked = source.obs.loc[embedding.obs_names, ["Sample", "Response"]].copy()
source.file.close()
assert embedding.n_obs == 29_614 and meta["Sample"].nunique() == 10
sample_key = meta.groupby("Sample", observed=True)[["Patient", "Tissue origin", "Platform"]].first().sort_index()
print("All embedded cells:", embedding.n_obs)
display(meta["Sample"].value_counts().reindex(sample_key.index).to_frame("whole_cells"))

## 1. Conventional line A — cell-state composition

Each sample is represented by published cell-state proportions. A centered
log-ratio (CLR) transform handles the fact that proportions sum to one. Response
is not used to select states or construct the representation.

In [ ]:
sample_ids = sample_key.index.tolist()
valid_state = meta["celltype"].notna() & ~meta["celltype"].astype(str).isin(["AMB cells", "NA", "nan"])
counts = pd.crosstab(
    meta.loc[valid_state, "Sample"].astype(str),
    meta.loc[valid_state, "celltype"].astype(str),
).reindex(sample_ids, fill_value=0)
proportions = counts.div(counts.sum(axis=1), axis=0)
pseudocount = 0.5 / counts.sum(axis=1)
adjusted = proportions.add(pseudocount, axis=0)
adjusted = adjusted.div(adjusted.sum(axis=1), axis=0)
composition_clr = np.log(adjusted).sub(np.log(adjusted).mean(axis=1), axis=0)
composition_matrix = composition_clr.to_numpy(dtype=np.float32)
composition_matrix /= np.maximum(np.linalg.norm(composition_matrix, axis=1, keepdims=True), 1e-12)
display(proportions.round(3))
print("Composition features:", composition_matrix.shape)

## 2. Conventional line B — CD4/CD8 functional programs

Predefined marker programs summarize Treg suppression, helper/memory, TH17,
cytotoxicity, exhaustion, interferon response and proliferation. Scores are
mean log-normalized expression within CD4/CD8 cells, aggregated per sample,
standardized across samples, and never selected using response.

In [ ]:
programs = {
    "Treg_suppression": ["FOXP3", "IL2RA", "CTLA4", "TIGIT"],
    "CD4_helper_memory": ["IL7R", "CCR7", "TCF7", "LTB"],
    "TH17": ["KLRB1", "CCR6", "RORA"],
    "cytotoxicity": ["NKG7", "CCL5", "PRF1", "GZMB", "GNLY"],
    "exhaustion": ["PDCD1", "TOX", "LAG3", "HAVCR2", "CXCL13"],
    "IFN_response": ["ISG15", "IFIT1", "IFIT3", "MX1"],
    "proliferation": ["MKI67", "TOP2A", "STMN1"],
}
cd48_ids = meta.index[meta["lineage.sub"].astype(str).isin(["CD4+ T cells", "CD8+ T cells"])]
raw = sc.read_h5ad(RAW_PATH)
available_genes = sorted({gene for genes in programs.values() for gene in genes if gene in raw.var_names})
functional_cells = raw[cd48_ids, available_genes].copy()
del raw
sc.pp.normalize_total(functional_cells, target_sum=1e4)
sc.pp.log1p(functional_cells)

cell_scores = pd.DataFrame(index=functional_cells.obs_names)
for program, genes in programs.items():
    genes = [gene for gene in genes if gene in functional_cells.var_names]
    cell_scores[program] = np.asarray(functional_cells[:, genes].X.mean(axis=1)).ravel()
cell_scores["Sample"] = meta.loc[cell_scores.index, "Sample"].astype(str)
functional_scores = cell_scores.groupby("Sample", observed=True)[list(programs)].mean().reindex(sample_ids)
functional_z = (functional_scores - functional_scores.mean()) / functional_scores.std(ddof=0).replace(0, 1)
functional_matrix = functional_z.to_numpy(dtype=np.float32)
functional_matrix /= np.maximum(np.linalg.norm(functional_matrix, axis=1, keepdims=True), 1e-12)
display(functional_scores.round(3))
print("Functional features:", functional_matrix.shape)

## 3. Foundation-model line — pooled scGPT embeddings

Mean pooling treats cells equally. Cluster-aware pooling gives globally rarer
unsupervised scGPT states up to 3× weight. Both are calculated for all cells
and for the combined CD4/CD8 compartment.

In [ ]:
def pool(obj, metadata, sample_ids):
    work = obj.copy()
    sc.pp.neighbors(work, use_rep="X", n_neighbors=15, random_state=RANDOM_STATE)
    sc.tl.leiden(work, resolution=0.3, key_added="cluster", random_state=RANDOM_STATE,
                 flavor="igraph", n_iterations=2, directed=False)
    matrix = np.asarray(work.X, dtype=np.float32)
    clusters = work.obs["cluster"].astype(str)
    frequency = clusters.value_counts(normalize=True)
    cell_weight = clusters.map((1 / np.sqrt(frequency)).clip(upper=3.0)).to_numpy()
    means, weighted = [], []
    for sample in sample_ids:
        selected = metadata["Sample"].astype(str).to_numpy() == sample
        assert selected.sum() > 0
        means.append(matrix[selected].mean(axis=0))
        weighted.append(np.average(matrix[selected], axis=0, weights=cell_weight[selected]))
    def norm(x):
        x = np.asarray(x, dtype=np.float32)
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    return norm(means), norm(weighted)

whole_mean, whole_weighted = pool(embedding, meta, sample_ids)
cd48_mask = meta["lineage.sub"].astype(str).isin(["CD4+ T cells", "CD8+ T cells"]).to_numpy()
cd48 = embedding[cd48_mask].copy()
cd48_meta = meta.iloc[np.flatnonzero(cd48_mask)].copy()
coverage = cd48_meta["Sample"].value_counts().reindex(sample_ids).fillna(0).astype(int)
assert coverage.gt(0).all()
display(coverage.to_frame("CD4_CD8_cells"))
cd48_mean, cd48_weighted = pool(cd48, cd48_meta, sample_ids)

representations = {
    "composition_CLR": composition_matrix,
    "CD4_CD8_programs": functional_matrix,
    "whole_mean": whole_mean,
    "whole_cluster_aware": whole_weighted,
    "CD4_CD8_mean": cd48_mean,
    "CD4_CD8_cluster_aware": cd48_weighted,
}
print({key: value.shape for key, value in representations.items()})

## 4. Reveal response and visualize all three evidence lines

In [ ]:
response = response_locked.groupby("Sample", observed=True)["Response"].first().reindex(sample_ids)
sample_key["Response"] = response.astype(str)
assert sample_key["Response"].value_counts().to_dict() == {"Non-responder": 7, "Responder": 3}
colors = {"Responder": "#2878B5", "Non-responder": "#D9534F"}
special_samples = set(sample_key.index[sample_key["Patient"].astype(str).eq("P1076")])
assert special_samples == {"EBUS_76", "NECK_05"}
titles = {
    "composition_CLR": "Conventional: cell-state composition (CLR)",
    "CD4_CD8_programs": "Conventional: CD4/CD8 functional programs",
    "whole_mean": "scGPT: whole cells — mean pooling",
    "whole_cluster_aware": "scGPT: whole cells — cluster-aware pooling",
    "CD4_CD8_mean": "scGPT: CD4/CD8 — mean pooling",
    "CD4_CD8_cluster_aware": "scGPT: CD4/CD8 — cluster-aware pooling",
}
umaps = {}
for name, matrix in representations.items():
    view = ad.AnnData(X=matrix, obs=sample_key.copy())
    sc.pp.neighbors(
        view, n_neighbors=3, use_rep="X", metric="cosine",
        random_state=RANDOM_STATE,
    )
    sc.tl.umap(view, random_state=RANDOM_STATE)
    umaps[name] = view.obsm["X_umap"].copy()
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for i, sample in enumerate(view.obs_names):
        response_group = view.obs.loc[sample, "Response"]
        marker = "^" if sample in special_samples else "o"
        ax.scatter(view.obsm["X_umap"][i, 0], view.obsm["X_umap"][i, 1],
                   s=105, color=colors[response_group], marker=marker,
                   edgecolor="white", linewidth=0.8)
        ax.annotate(sample, view.obsm["X_umap"][i], xytext=(4, 4), textcoords="offset points", fontsize=8)
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=colors["Responder"],
               markeredgecolor="white", markersize=9, label="Responder"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor=colors["Non-responder"],
               markeredgecolor="white", markersize=9, label="Non-responder"),
        Line2D([0], [0], marker="^", color="none", markerfacecolor="#777777",
               markeredgecolor="white", markersize=9, label="P1076 samples"),
    ]
    ax.set(title=titles[name], xlabel="UMAP1", ylabel="UMAP2")
    ax.legend(handles=handles, frameon=False); fig.tight_layout()
    fig.savefig(FIGURES / f"sample_{name}_response_umap.png", dpi=240, bbox_inches="tight")
    plt.show()

## 5. Distance evidence across the three lines

In [ ]:
def pair_mean(distance, left, right=None):
    left = np.flatnonzero(left)
    pairs = list(combinations(left, 2)) if right is None else [(i, j) for i in left for j in np.flatnonzero(right)]
    return float(np.mean([distance[i, j] for i, j in pairs]))

labels = sample_key["Response"].to_numpy()
rows = []
for name, matrix in representations.items():
    d = pairwise_distances(matrix, metric="cosine")
    r = labels == "Responder"; n = ~r
    centroids = np.vstack([matrix[r].mean(0), matrix[n].mean(0)])
    observed = silhouette_score(matrix, labels, metric="cosine")
    null = []
    for chosen in combinations(range(10), 3):
        perm = np.array(["Non-responder"] * 10, dtype=object); perm[list(chosen)] = "Responder"
        null.append(silhouette_score(matrix, perm, metric="cosine"))
    rows.append({"representation": name, "silhouette_cosine": observed,
                 "within_responder": pair_mean(d, r), "within_nonresponder": pair_mean(d, n),
                 "between_groups": pair_mean(d, r, n),
                 "centroid_distance": pairwise_distances(centroids, metric="cosine")[0, 1],
                 "exact_10sample_permutation_p": np.mean(np.asarray(null) >= observed),
                 "n_permutations": len(null)})
metrics = pd.DataFrame(rows).sort_values("silhouette_cosine", ascending=False)
display(metrics)
metrics.to_csv(TABLES / "sample_level_three_line_metrics.csv", index=False)

direction_rows = []
response_mask = sample_key["Response"].eq("Responder").to_numpy()
for line, frame in {
    "composition_CLR": composition_clr,
    "CD4_CD8_programs": functional_z,
}.items():
    for feature in frame.columns:
        responder_mean = float(frame.loc[response_mask, feature].mean())
        nonresponder_mean = float(frame.loc[~response_mask, feature].mean())
        direction_rows.append({
            "line": line, "feature": feature,
            "responder_mean": responder_mean,
            "nonresponder_mean": nonresponder_mean,
            "responder_minus_nonresponder": responder_mean - nonresponder_mean,
        })
feature_directions = pd.DataFrame(direction_rows)
display(feature_directions.loc[feature_directions["line"].eq("CD4_CD8_programs")]
        .sort_values("responder_minus_nonresponder", ascending=False))
display(feature_directions.loc[feature_directions["line"].eq("composition_CLR")]
        .assign(abs_difference=lambda x: x["responder_minus_nonresponder"].abs())
        .sort_values("abs_difference", ascending=False).head(12).drop(columns="abs_difference"))
feature_directions.to_csv(TABLES / "sample_level_feature_directions.csv", index=False)

## 6. Save reusable sample representations

In [ ]:
out = ad.AnnData(X=whole_mean, obs=sample_key.copy())
out.var_names = embedding.var_names.copy()
out.obsm["X_whole_cluster_aware"] = whole_weighted
out.obsm["X_CD4_CD8_mean"] = cd48_mean
out.obsm["X_CD4_CD8_cluster_aware"] = cd48_weighted
out.obsm["X_composition_CLR"] = composition_matrix
out.obsm["X_CD4_CD8_programs"] = functional_matrix
out.uns["composition_features"] = composition_clr.columns.astype(str).tolist()
out.uns["functional_programs"] = list(programs)
for name, coords in umaps.items(): out.obsm[f"X_umap_{name}"] = coords
path = EMBEDDINGS / "GSE205335_10sample_sample_embeddings_v1.h5ad"
out.write_h5ad(path, compression="gzip")
print("Saved", path)

## 7. Demo conclusion

Compare whether response is visible through cell composition, through
interpretable CD4/CD8 functions, and through frozen scGPT geometry. Agreement
across lines is stronger evidence than one attractive UMAP. With ten samples
this remains an explanatory demo, not a validated predictor.